# Perplexity: Comprehensive Guide for LLM Evaluation

#### Overview
Perplexity is a fundamental metric used to evaluate language models by measuring how well a probability model predicts a sample. This notebook provides a comprehensive guide to understanding, implementing, and applying perplexity in LLM evaluation and comparison.

#### Table of Contents
1. [Import Required Libraries](#section1)
2. [Understanding Perplexity](#section2)  
3. [Perplexity Use Cases and Applications](#section3)
4. [When to Apply Perplexity vs Other Metrics](#section4)
5. [Pros and Cons Analysis](#section5)
6. [Basic Perplexity Implementation](#section6)
7. [Perplexity for Language Model Evaluation](#section7)
8. [Perplexity in Text Generation Quality](#section8)
9. [Advanced Perplexity Calculations](#section9)
10. [Comparing Multiple LLMs with Perplexity](#section10)

#### 1. Import Required Libraries {#section1}

Let's start by importing all the necessary libraries for perplexity calculation and LLM evaluation.

In [4]:
import langflow
import os
print(os.path.dirname(langflow.__file__))

d:\Makesh\Working\AI\Python_Envs\gen_ai\Lib\site-packages\langflow


In [2]:
!pip install entropy

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for entropy: filename=entropy-0.1.5-py3-none-any.whl size=7266 sha256=533a4b7f024ed6293c5180f5b163be72ecb2c68a263a0062ec9cb856eebfbf06
  Stored in directory: c:\users\reach\appdata\local\pip\cache\wheels\e7\d7\1a\b069c8c162d581350b161d7be566da204e1957a17e6eaeddd0
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-no

In [3]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
import warnings
warnings.filterwarnings('ignore')

# For text processing
import re
from collections import Counter, defaultdict

# For advanced perplexity calculations
try:
    import torch
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, Dataset
except ImportError:
    print("Installing PyTorch for advanced perplexity calculations...")
    import subprocess
    subprocess.check_call(["pip", "install", "torch", "torchvision", "torchaudio"])
    import torch
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, Dataset

# For LLM integration and tokenization
try:
    from transformers import (
        AutoTokenizer, AutoModelForCausalLM, 
        GPT2LMHeadModel, GPT2TokenizerFast,
        pipeline
    )
except ImportError:
    print("Installing transformers for LLM evaluation...")
    import subprocess
    subprocess.check_call(["pip", "install", "transformers"])
    from transformers import (
        AutoTokenizer, AutoModelForCausalLM, 
        GPT2LMHeadModel, GPT2TokenizerFast,
        pipeline
    )

# For statistical analysis
try:
    from scipy import stats
    from scipy.special import entropy
except ImportError:
    print("Installing scipy for statistical analysis...")
    import subprocess
    subprocess.check_call(["pip", "install", "scipy"])
    from scipy import stats
    from scipy.special import entropy

# NLP libraries
try:
    import nltk
    from nltk.tokenize import word_tokenize, sent_tokenize
    from nltk.corpus import stopwords
    from nltk.util import ngrams
except ImportError:
    print("Installing nltk...")
    import subprocess
    subprocess.check_call(["pip", "install", "nltk"])
    import nltk
    from nltk.tokenize import word_tokenize, sent_tokenize
    from nltk.corpus import stopwords
    from nltk.util import ngrams

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('punkt')
    nltk.download('stopwords')

print("All libraries imported successfully!")
print("PyTorch version:", torch.__version__ if 'torch' in globals() else "Not available")

Installing scipy for statistical analysis...


ImportError: cannot import name 'entropy' from 'scipy.special' (d:\Makesh\Working\AI\Python_Envs\gen_ai\Lib\site-packages\scipy\special\__init__.py)

#### 2. Understanding Perplexity {#section2}

Perplexity measures how well a probability model predicts a test set. In the context of language models, it quantifies how "surprised" the model is by the text it's trying to predict.

**Mathematical Foundation:**

For a sequence of tokens $w_1, w_2, ..., w_N$:

$$\text{Perplexity}(W) = \sqrt[N]{\frac{1}{P(w_1, w_2, ..., w_N)}}$$

Or equivalently:

$$\text{Perplexity}(W) = 2^{H(W)}$$

Where $H(W)$ is the cross-entropy:

$$H(W) = -\frac{1}{N} \sum_{i=1}^{N} \log_2 P(w_i | w_1, ..., w_{i-1})$$

**Key Concepts:**
- **Lower perplexity** = Better model (less surprised)
- **Higher perplexity** = Worse model (more surprised)
- **Baseline**: Random model has perplexity equal to vocabulary size
- **Perfect model**: Perplexity = 1 (never surprised)

In [ ]:
# Basic perplexity calculation demonstration
def calculate_basic_perplexity(probabilities):
    """
    Calculate perplexity from a list of probabilities
    
    Args:
        probabilities: List of probabilities for each token
    
    Returns:
        float: Perplexity value
    """
    # Convert to numpy array for easier computation
    probs = np.array(probabilities)
    
    # Avoid log(0) by adding small epsilon
    epsilon = 1e-10
    probs = np.maximum(probs, epsilon)
    
    # Calculate cross-entropy
    cross_entropy = -np.mean(np.log2(probs))
    
    # Calculate perplexity
    perplexity = 2 ** cross_entropy
    
    return perplexity, cross_entropy

def demonstrate_perplexity_intuition():
    """Demonstrate perplexity intuition with examples"""
    
    print("=== Perplexity Intuition Demonstration ===")
    
    # Example 1: Perfect predictions
    perfect_probs = [1.0, 1.0, 1.0, 1.0, 1.0]
    perp1, ce1 = calculate_basic_perplexity(perfect_probs)
    print(f"Perfect predictions (all prob=1.0): Perplexity = {perp1:.4f}, Cross-entropy = {ce1:.4f}")
    
    # Example 2: Good predictions
    good_probs = [0.9, 0.85, 0.8, 0.9, 0.88]
    perp2, ce2 = calculate_basic_perplexity(good_probs)
    print(f"Good predictions (prob~0.85): Perplexity = {perp2:.4f}, Cross-entropy = {ce2:.4f}")
    
    # Example 3: Poor predictions
    poor_probs = [0.3, 0.25, 0.4, 0.35, 0.2]
    perp3, ce3 = calculate_basic_perplexity(poor_probs)
    print(f"Poor predictions (prob~0.3): Perplexity = {perp3:.4f}, Cross-entropy = {ce3:.4f}")
    
    # Example 4: Random predictions (uniform over vocab of size 1000)
    random_probs = [0.001] * 5  # 1/1000 for each token
    perp4, ce4 = calculate_basic_perplexity(random_probs)
    print(f"Random predictions (vocab=1000): Perplexity = {perp4:.4f}, Cross-entropy = {ce4:.4f}")
    
    print("\\n" + "="*60)
    print("Key Insight: Lower perplexity indicates better model performance!")
    print("="*60)
    
    # Visualization
    scenarios = ['Perfect', 'Good', 'Poor', 'Random']
    perplexities = [perp1, perp2, perp3, perp4]
    cross_entropies = [ce1, ce2, ce3, ce4]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Perplexity comparison
    bars1 = ax1.bar(scenarios, perplexities, color=['green', 'blue', 'orange', 'red'], alpha=0.7)
    ax1.set_ylabel('Perplexity')
    ax1.set_title('Perplexity Comparison')
    ax1.set_yscale('log')
    
    # Add value labels on bars
    for bar, val in zip(bars1, perplexities):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.1, 
                f'{val:.2f}', ha='center', va='bottom')
    
    # Cross-entropy comparison
    bars2 = ax2.bar(scenarios, cross_entropies, color=['green', 'blue', 'orange', 'red'], alpha=0.7)
    ax2.set_ylabel('Cross-entropy (bits)')
    ax2.set_title('Cross-entropy Comparison')
    
    # Add value labels on bars
    for bar, val in zip(bars2, cross_entropies):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
                f'{val:.2f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
    
    return {
        'scenarios': scenarios,
        'perplexities': perplexities,
        'cross_entropies': cross_entropies
    }

# Run demonstration
intuition_results = demonstrate_perplexity_intuition()

#### 3. Perplexity Use Cases and Applications {#section3}

Perplexity is widely used across various NLP tasks and model evaluation scenarios:

In [ ]:
def demonstrate_perplexity_use_cases():
    """Demonstrate perplexity applications across different scenarios"""
    
    print("=== Perplexity Use Cases Demonstration ===")
    
    # Use Case 1: Language Model Comparison
    print("\\n**Use Case 1: Language Model Comparison**")
    print("-" * 50)
    
    # Simulate different model performances on the same text
    test_sentence = "The quick brown fox jumps over the lazy dog"
    
    # Simulate probability distributions from different models
    models_performance = {
        "GPT-4": [0.85, 0.90, 0.88, 0.92, 0.87, 0.89, 0.91, 0.86, 0.88],
        "GPT-3.5": [0.75, 0.80, 0.78, 0.82, 0.77, 0.79, 0.81, 0.76, 0.78],
        "BERT": [0.65, 0.70, 0.68, 0.72, 0.67, 0.69, 0.71, 0.66, 0.68],
        "Basic-LM": [0.45, 0.50, 0.48, 0.52, 0.47, 0.49, 0.51, 0.46, 0.48]
    }
    
    model_results = {}
    for model_name, probs in models_performance.items():
        perp, ce = calculate_basic_perplexity(probs)
        model_results[model_name] = {'perplexity': perp, 'cross_entropy': ce}
        print(f"{model_name:12}: Perplexity = {perp:6.2f}, Cross-entropy = {ce:5.2f} bits")
    
    print(f"\\nTest sentence: '{test_sentence}'")
    print("Lower perplexity indicates better language modeling capability.")
    
    # Use Case 2: Domain Adaptation Assessment
    print("\\n\\n**Use Case 2: Domain Adaptation Assessment**")
    print("-" * 50)
    
    domains = {
        "General Text": [0.75, 0.80, 0.78, 0.82, 0.77],
        "Medical Text": [0.45, 0.50, 0.48, 0.52, 0.47],  # Model struggles with domain-specific terms
        "Legal Text": [0.40, 0.45, 0.43, 0.47, 0.42],   # Even more specialized
        "Conversational": [0.85, 0.88, 0.86, 0.90, 0.87]  # Model trained on dialogue
    }
    
    domain_results = {}
    for domain, probs in domains.items():
        perp, ce = calculate_basic_perplexity(probs)
        domain_results[domain] = {'perplexity': perp, 'cross_entropy': ce}
        print(f"{domain:15}: Perplexity = {perp:6.2f}")
    
    print("\\nHigher perplexity on specialized domains indicates need for domain adaptation.")
    
    # Use Case 3: Text Quality Assessment
    print("\\n\\n**Use Case 3: Generated Text Quality Assessment**")
    print("-" * 50)
    
    generated_texts = {
        "Human-written": [0.88, 0.85, 0.90, 0.87, 0.89, 0.86, 0.91],
        "High-quality AI": [0.82, 0.79, 0.84, 0.81, 0.83, 0.80, 0.85],
        "Medium-quality AI": [0.65, 0.62, 0.68, 0.64, 0.67, 0.63, 0.69],
        "Low-quality AI": [0.35, 0.32, 0.38, 0.34, 0.37, 0.33, 0.39],
        "Completely Random": [0.001] * 7  # Random selection from large vocabulary
    }
    
    quality_results = {}
    for text_type, probs in generated_texts.items():
        perp, ce = calculate_basic_perplexity(probs)
        quality_results[text_type] = {'perplexity': perp, 'cross_entropy': ce}
        print(f"{text_type:18}: Perplexity = {perp:8.2f}")
    
    print("\\nLower perplexity often correlates with higher text quality and naturalness.")
    
    # Use Case 4: Training Progress Monitoring
    print("\\n\\n**Use Case 4: Training Progress Monitoring**")
    print("-" * 50)
    
    # Simulate perplexity during training epochs
    training_epochs = list(range(1, 11))
    train_perplexity = [150, 120, 95, 78, 65, 58, 53, 50, 48, 47]
    val_perplexity = [165, 135, 110, 92, 80, 75, 72, 71, 72, 74]  # Shows overfitting
    
    print("Epoch | Train PPL | Val PPL | Status")
    print("-" * 35)
    for epoch, train_ppl, val_ppl in zip(training_epochs, train_perplexity, val_perplexity):
        status = "Good" if val_ppl <= train_ppl * 1.1 else "Overfitting"
        print(f"{epoch:5} | {train_ppl:9.1f} | {val_ppl:7.1f} | {status}")
    
    # Visualization
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
    
    # 1. Model Comparison
    models = list(model_results.keys())
    model_perps = [model_results[m]['perplexity'] for m in models]
    bars1 = ax1.bar(models, model_perps, color=['green', 'blue', 'orange', 'red'], alpha=0.7)
    ax1.set_ylabel('Perplexity')
    ax1.set_title('Language Model Comparison')
    ax1.tick_params(axis='x', rotation=45)
    
    # Add value labels
    for bar, val in zip(bars1, model_perps):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                f'{val:.2f}', ha='center', va='bottom')
    
    # 2. Domain Adaptation
    domains_list = list(domain_results.keys())
    domain_perps = [domain_results[d]['perplexity'] for d in domains_list]
    bars2 = ax2.bar(domains_list, domain_perps, color='skyblue', alpha=0.7)
    ax2.set_ylabel('Perplexity')
    ax2.set_title('Domain-specific Performance')
    ax2.tick_params(axis='x', rotation=45)
    
    # 3. Text Quality
    quality_types = list(quality_results.keys())
    quality_perps = [quality_results[q]['perplexity'] for q in quality_types]
    bars3 = ax3.bar(quality_types, quality_perps, color='lightcoral', alpha=0.7)
    ax3.set_ylabel('Perplexity (log scale)')
    ax3.set_yscale('log')
    ax3.set_title('Generated Text Quality Assessment')
    ax3.tick_params(axis='x', rotation=45)
    
    # 4. Training Progress
    ax4.plot(training_epochs, train_perplexity, 'b-o', label='Training', linewidth=2)
    ax4.plot(training_epochs, val_perplexity, 'r-s', label='Validation', linewidth=2)
    ax4.set_xlabel('Epoch')
    ax4.set_ylabel('Perplexity')
    ax4.set_title('Training Progress Monitoring')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return {
        'models': model_results,
        'domains': domain_results,
        'quality': quality_results,
        'training': {'epochs': training_epochs, 'train': train_perplexity, 'val': val_perplexity}
    }

# Run use cases demonstration
use_case_results = demonstrate_perplexity_use_cases()

#### 4. When to Apply Perplexity vs Other Metrics {#section4}

Understanding when to use perplexity versus other evaluation metrics is crucial for proper model assessment:

In [ ]:
def compare_evaluation_metrics():
    """Compare perplexity with other evaluation metrics across different scenarios"""
    
    # Sample text for evaluation
    test_text = "Artificial intelligence is transforming the world through innovative applications."
    
    # Simulate different model outputs and their characteristics
    evaluation_scenarios = {
        "High-quality output": {
            "text": "AI is revolutionizing the world with innovative technological applications.",
            "perplexity": 12.5,
            "bleu_score": 0.75,
            "rouge_score": 0.80,
            "human_rating": 4.2
        },
        "Medium-quality output": {
            "text": "Artificial intelligence changes world through new applications and technology.",
            "perplexity": 25.8,
            "bleu_score": 0.45,
            "rouge_score": 0.60,
            "human_rating": 3.1
        },
        "Low-quality output": {
            "text": "Intelligence artificial world transforming applications innovative through is.",
            "perplexity": 145.2,
            "bleu_score": 0.15,
            "rouge_score": 0.25,
            "human_rating": 1.8
        },
        "Fluent but incorrect": {
            "text": "Artificial intelligence is destroying the world through dangerous applications.",
            "perplexity": 15.3,  # Low perplexity (fluent) but wrong content
            "bleu_score": 0.65,
            "rouge_score": 0.70,
            "human_rating": 2.0   # Low rating due to incorrect content
        }
    }
    
    print("=== Metric Comparison Analysis ===")
    print(f"Reference text: {test_text}")
    print("\\n" + "="*80)
    
    # Create comparison table
    comparison_data = []
    
    for scenario, metrics in evaluation_scenarios.items():
        print(f"\\n{scenario}:")
        print(f"  Generated: {metrics['text']}")
        print(f"  Perplexity: {metrics['perplexity']:6.1f}")
        print(f"  BLEU Score: {metrics['bleu_score']:6.2f}")
        print(f"  ROUGE Score: {metrics['rouge_score']:5.2f}")
        print(f"  Human Rating: {metrics['human_rating']:4.1f}/5.0")
        
        comparison_data.append({
            'Scenario': scenario,
            'Perplexity': metrics['perplexity'],
            'BLEU': metrics['bleu_score'],
            'ROUGE': metrics['rouge_score'],
            'Human_Rating': metrics['human_rating']
        })
    
    # Convert to DataFrame for analysis
    df = pd.DataFrame(comparison_data)
    
    print("\\n" + "="*80)
    print("CORRELATION ANALYSIS:")
    print("-" * 40)
    
    # Calculate correlations with human ratings
    perp_corr = df['Perplexity'].corr(df['Human_Rating'])
    bleu_corr = df['BLEU'].corr(df['Human_Rating'])
    rouge_corr = df['ROUGE'].corr(df['Human_Rating'])
    
    print(f"Perplexity vs Human Rating: {perp_corr:6.3f} (negative correlation expected)")
    print(f"BLEU vs Human Rating:       {bleu_corr:6.3f}")
    print(f"ROUGE vs Human Rating:      {rouge_corr:6.3f}")
    
    print("\\nKey Insight: The 'Fluent but incorrect' case shows perplexity limitations!")
    
    # Visualizations
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # 1. Metric comparison bar chart
    scenarios = df['Scenario']
    x_pos = np.arange(len(scenarios))
    
    # Normalize perplexity for visualization (invert and scale)
    norm_perplexity = 1 / (df['Perplexity'] / 100)  # Invert so lower is better
    
    ax1 = axes[0, 0]
    width = 0.2
    ax1.bar(x_pos - 1.5*width, norm_perplexity, width, label='Perplexity (inv)', alpha=0.8)
    ax1.bar(x_pos - 0.5*width, df['BLEU'], width, label='BLEU', alpha=0.8)
    ax1.bar(x_pos + 0.5*width, df['ROUGE'], width, label='ROUGE', alpha=0.8)
    ax1.bar(x_pos + 1.5*width, df['Human_Rating']/5, width, label='Human (scaled)', alpha=0.8)
    
    ax1.set_xlabel('Scenarios')
    ax1.set_ylabel('Normalized Scores')
    ax1.set_title('Metric Comparison Across Scenarios')
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels([s.replace(' ', '\\n') for s in scenarios], fontsize=8)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. Perplexity vs Human Rating scatter
    ax2 = axes[0, 1]
    colors = ['green', 'blue', 'red', 'orange']
    for i, (_, row) in enumerate(df.iterrows()):
        ax2.scatter(row['Perplexity'], row['Human_Rating'], 
                   c=colors[i], s=100, alpha=0.7, label=row['Scenario'])
    
    ax2.set_xlabel('Perplexity')
    ax2.set_ylabel('Human Rating')
    ax2.set_title('Perplexity vs Human Judgment')
    ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax2.grid(True, alpha=0.3)
    
    # 3. All metrics vs Human Rating
    ax3 = axes[1, 0]
    ax3.scatter(df['BLEU'], df['Human_Rating'], label='BLEU', alpha=0.7, s=80)
    ax3.scatter(df['ROUGE'], df['Human_Rating'], label='ROUGE', alpha=0.7, s=80)
    ax3.scatter(1/df['Perplexity']*50, df['Human_Rating'], label='1/Perplexity (scaled)', alpha=0.7, s=80)
    
    ax3.set_xlabel('Metric Score')
    ax3.set_ylabel('Human Rating')
    ax3.set_title('Various Metrics vs Human Judgment')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # 4. Metric agreement heatmap
    ax4 = axes[1, 1]
    # Create correlation matrix
    metric_df = df[['Perplexity', 'BLEU', 'ROUGE', 'Human_Rating']].copy()
    metric_df['Perplexity'] = 1 / metric_df['Perplexity']  # Invert for positive correlation
    
    corr_matrix = metric_df.corr()
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, ax=ax4, 
                square=True, fmt='.3f')
    ax4.set_title('Metric Correlation Matrix')
    
    plt.tight_layout()
    plt.show()
    
    return df

# Run metric comparison
metric_comparison_df = compare_evaluation_metrics()

**When to Use Perplexity vs Other Metrics:**

| Metric | Best For | Limitations | Perplexity Comparison |
|--------|----------|-------------|----------------------|
| **Perplexity** | Language model evaluation, fluency assessment | Doesn't measure semantic correctness | Baseline metric |
| **BLEU** | Machine translation, exact match tasks | Poor for diverse valid outputs | Complementary for precision |
| **ROUGE** | Summarization, content overlap | Doesn't capture semantic similarity | Complementary for recall |
| **METEOR** | Translation with synonyms | More complex computation | Better semantic awareness |
| **BERTScore** | Semantic similarity | Computationally expensive | Modern semantic alternative |
| **Human Evaluation** | Overall quality assessment | Expensive and time-consuming | Gold standard for validation |

**Key Decision Framework:**
- **Use Perplexity when**: Evaluating language model quality, measuring fluency, comparing model architectures
- **Combine with other metrics when**: Assessing task-specific performance, measuring semantic correctness
- **Avoid Perplexity alone when**: Content accuracy is critical, semantic similarity matters more than fluency

#### 5. Pros and Cons Analysis {#section5}

Let's examine the strengths and limitations of perplexity through practical examples:

In [ ]:
def analyze_perplexity_pros_cons():
    """Demonstrate perplexity strengths and limitations through examples"""
    
    print("=== Perplexity Pros and Cons Analysis ===")
    
    # Test cases to highlight perplexity behavior
    test_cases = [
        {
            "name": "Fluent and Correct",
            "text": "The weather today is sunny and warm.",
            "probabilities": [0.85, 0.90, 0.88, 0.82, 0.87, 0.89, 0.86],
            "scenario": "✅ STRENGTH: Correctly identifies high-quality text"
        },
        {
            "name": "Fluent but Factually Wrong",
            "text": "The sun revolves around the Earth daily.",
            "probabilities": [0.83, 0.88, 0.85, 0.84, 0.86, 0.87, 0.85],  # Still fluent!
            "scenario": "❌ LIMITATION: Cannot detect factual incorrectness"
        },
        {
            "name": "Grammatically Poor",
            "text": "Weather sunny today is very and warm.",
            "probabilities": [0.65, 0.45, 0.70, 0.35, 0.55, 0.60, 0.50],
            "scenario": "✅ STRENGTH: Correctly penalizes poor grammar"
        },
        {
            "name": "Technical Jargon (In-domain)",
            "text": "The mitochondria generates ATP through oxidative phosphorylation.",
            "probabilities": [0.80, 0.75, 0.82, 0.78, 0.76, 0.79, 0.81],  # Model trained on scientific text
            "scenario": "✅ STRENGTH: Recognizes domain-appropriate language"
        },
        {
            "name": "Technical Jargon (Out-of-domain)",
            "text": "The mitochondria generates ATP through oxidative phosphorylation.",
            "probabilities": [0.40, 0.25, 0.45, 0.30, 0.35, 0.38, 0.32],  # Model not trained on scientific text
            "scenario": "❌ LIMITATION: Domain dependency affects scores"
        },
        {
            "name": "Repetitive but Fluent",
            "text": "The cat sat on the mat. The cat sat on the mat.",
            "probabilities": [0.88, 0.85, 0.90, 0.87, 0.89, 0.88, 0.85, 0.90, 0.87, 0.89, 0.88, 0.85],
            "scenario": "❌ LIMITATION: May not penalize excessive repetition"
        },
        {
            "name": "Creative but Unusual",
            "text": "The kaleidoscope butterfly whispered secrets to moonbeams.",
            "probabilities": [0.45, 0.35, 0.40, 0.30, 0.38, 0.42, 0.36, 0.44],
            "scenario": "❌ LIMITATION: Penalizes creativity and unusual expressions"
        }
    ]
    
    results = []
    
    for case in test_cases:
        perp, ce = calculate_basic_perplexity(case['probabilities'])
        
        print(f"\\n{case['name']}:")
        print(f"  Text: '{case['text']}'")
        print(f"  Perplexity: {perp:.2f}")
        print(f"  Scenario: {case['scenario']}")
        
        results.append({
            'name': case['name'],
            'perplexity': perp,
            'cross_entropy': ce,
            'category': case['scenario'].split(':')[0].strip()
        })
    
    # Additional analysis: Perplexity vs text length
    print("\\n" + "="*70)
    print("LENGTH DEPENDENCY ANALYSIS:")
    print("-" * 30)
    
    length_tests = [
        {"text": "Good.", "probs": [0.85]},
        {"text": "This is good.", "probs": [0.80, 0.85, 0.82]},
        {"text": "This is a very good example.", "probs": [0.80, 0.85, 0.82, 0.78, 0.83, 0.81]},
        {"text": "This is a very good example of high quality text.", "probs": [0.80, 0.85, 0.82, 0.78, 0.83, 0.81, 0.79, 0.84, 0.80]}
    ]
    
    for test in length_tests:
        perp, _ = calculate_basic_perplexity(test['probs'])
        word_count = len(test['text'].split())
        print(f"  {word_count:2d} words: Perplexity = {perp:5.2f} | '{test['text']}'")
    
    print("\\nNote: Perplexity can be affected by sequence length due to error accumulation.")
    
    # Visualization
    df = pd.DataFrame(results)
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. Perplexity by scenario type
    strength_cases = df[df['category'] == '✅ STRENGTH']
    limitation_cases = df[df['category'] == '❌ LIMITATION']
    
    ax1.bar(range(len(strength_cases)), strength_cases['perplexity'], 
           color='green', alpha=0.7, label='Strengths', width=0.4)
    ax1.bar(range(len(strength_cases), len(strength_cases) + len(limitation_cases)), 
           limitation_cases['perplexity'], 
           color='red', alpha=0.7, label='Limitations', width=0.4)
    
    ax1.set_ylabel('Perplexity')
    ax1.set_title('Perplexity Strengths vs Limitations')
    ax1.set_xticks(range(len(df)))
    ax1.set_xticklabels([name.replace(' ', '\\n') for name in df['name']], 
                       rotation=45, ha='right', fontsize=9)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. Cross-entropy distribution
    ax2.hist(strength_cases['cross_entropy'], bins=5, alpha=0.7, color='green', 
            label='Strengths', density=True)
    ax2.hist(limitation_cases['cross_entropy'], bins=5, alpha=0.7, color='red', 
            label='Limitations', density=True)
    ax2.set_xlabel('Cross-entropy (bits)')
    ax2.set_ylabel('Density')
    ax2.set_title('Cross-entropy Distribution')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. Length dependency visualization
    lengths = [len(test['text'].split()) for test in length_tests]
    length_perps = [calculate_basic_perplexity(test['probs'])[0] for test in length_tests]
    
    ax3.plot(lengths, length_perps, 'bo-', linewidth=2, markersize=8)
    ax3.set_xlabel('Text Length (words)')
    ax3.set_ylabel('Perplexity')
    ax3.set_title('Perplexity vs Text Length')
    ax3.grid(True, alpha=0.3)
    
    # Add trend line
    z = np.polyfit(lengths, length_perps, 1)
    p = np.poly1d(z)
    ax3.plot(lengths, p(lengths), "r--", alpha=0.8, label=f'Trend: slope={z[0]:.3f}')
    ax3.legend()
    
    # 4. Scenario comparison radar chart
    categories = ['Fluency', 'Factual Accuracy', 'Grammar', 'Domain Adaptation', 'Creativity']
    perplexity_performance = [4, 2, 4, 3, 2]  # Subjective ratings out of 5
    ideal_performance = [5, 5, 5, 5, 5]
    
    angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
    angles += angles[:1]  # Complete the circle
    
    perplexity_performance += perplexity_performance[:1]
    ideal_performance += ideal_performance[:1]
    
    ax4.plot(angles, perplexity_performance, 'o-', linewidth=2, label='Perplexity', color='blue')
    ax4.fill(angles, perplexity_performance, alpha=0.25, color='blue')
    ax4.plot(angles, ideal_performance, 'o-', linewidth=2, label='Ideal Metric', color='green')
    ax4.fill(angles, ideal_performance, alpha=0.1, color='green')
    
    ax4.set_xticks(angles[:-1])
    ax4.set_xticklabels(categories)
    ax4.set_ylim(0, 5)
    ax4.set_title('Perplexity Performance Profile')
    ax4.legend(loc='upper right', bbox_to_anchor=(1.2, 1.0))
    ax4.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    return df

# Run pros and cons analysis
pros_cons_df = analyze_perplexity_pros_cons()

**Perplexity Pros:**
✅ **Intrinsic evaluation metric - no reference needed**  
✅ **Directly measures model confidence and fluency**  
✅ **Fast and efficient computation**  
✅ **Language and task agnostic**  
✅ **Good for model comparison and selection**  
✅ **Reflects training objective for most LMs**  
✅ **Sensitive to grammatical correctness**

**Perplexity Cons:**
❌ **Doesn't measure semantic correctness or factual accuracy**  
❌ **Domain and training data dependent**  
❌ **Can be gamed with repetitive or formulaic text**  
❌ **Penalizes creative and novel expressions**  
❌ **Sensitive to vocabulary size and tokenization**  
❌ **Doesn't correlate perfectly with human judgments**  
❌ **Limited for task-specific evaluation**

#### 6. Basic Perplexity Implementation {#section6}

Let's build a custom perplexity calculator from scratch to understand the underlying algorithms:

In [ ]:
class CustomPerplexityCalculator:
    """Custom implementation of perplexity calculation"""
    
    def __init__(self, smoothing_method='add_one', vocab_size=10000):
        self.smoothing_method = smoothing_method
        self.vocab_size = vocab_size
        self.epsilon = 1e-10  # Small value to avoid log(0)
    
    def tokenize(self, text):
        """Simple tokenization"""
        # Convert to lowercase and split on whitespace
        tokens = text.lower().replace('.', '').replace(',', '').replace('!', '').replace('?', '').split()
        return tokens
    
    def build_ngram_model(self, texts, n=2):
        """Build n-gram language model from training texts"""
        ngram_counts = defaultdict(int)
        context_counts = defaultdict(int)
        
        for text in texts:
            tokens = self.tokenize(text)
            # Add start and end tokens
            padded_tokens = ['<s>'] * (n-1) + tokens + ['</s>']
            
            for i in range(len(padded_tokens) - n + 1):
                ngram = tuple(padded_tokens[i:i+n])
                context = ngram[:-1]
                
                ngram_counts[ngram] += 1
                context_counts[context] += 1
        
        return ngram_counts, context_counts
    
    def calculate_probability(self, ngram, ngram_counts, context_counts):
        """Calculate probability of n-gram with smoothing"""
        context = ngram[:-1]
        
        if self.smoothing_method == 'add_one':
            # Add-one (Laplace) smoothing
            numerator = ngram_counts[ngram] + 1
            denominator = context_counts[context] + self.vocab_size
            return numerator / denominator
        
        elif self.smoothing_method == 'mle':
            # Maximum Likelihood Estimation (no smoothing)
            if context_counts[context] == 0:
                return self.epsilon
            return ngram_counts[ngram] / context_counts[context]
        
        else:
            raise ValueError(f"Unknown smoothing method: {self.smoothing_method}")
    
    def calculate_perplexity(self, test_text, ngram_counts, context_counts, n=2):
        """Calculate perplexity on test text"""
        tokens = self.tokenize(test_text)
        padded_tokens = ['<s>'] * (n-1) + tokens + ['</s>']
        
        log_prob_sum = 0
        token_count = 0
        
        for i in range(len(padded_tokens) - n + 1):
            ngram = tuple(padded_tokens[i:i+n])
            prob = self.calculate_probability(ngram, ngram_counts, context_counts)
            log_prob_sum += math.log2(max(prob, self.epsilon))
            token_count += 1
        
        # Calculate average log probability
        avg_log_prob = log_prob_sum / token_count
        
        # Calculate perplexity
        perplexity = 2 ** (-avg_log_prob)
        
        return perplexity, avg_log_prob, token_count
    
    def evaluate_model(self, train_texts, test_texts, n=2):
        """Evaluate model on test set"""
        print(f"Building {n}-gram model...")
        ngram_counts, context_counts = self.build_ngram_model(train_texts, n)
        
        print(f"Vocabulary size: {len(set(' '.join(train_texts).split()))}")
        print(f"Total {n}-grams: {len(ngram_counts)}")
        print(f"Smoothing method: {self.smoothing_method}")
        print()
        
        results = []
        for i, test_text in enumerate(test_texts):
            perp, avg_log_prob, tokens = self.calculate_perplexity(
                test_text, ngram_counts, context_counts, n)
            
            print(f"Test {i+1}: '{test_text[:50]}...' (Perplexity: {perp:.2f})")
            
            results.append({
                'text': test_text,
                'perplexity': perp,
                'avg_log_prob': avg_log_prob,
                'token_count': tokens
            })
        
        return results, ngram_counts, context_counts

def demonstrate_custom_perplexity():
    """Demonstrate custom perplexity implementation"""
    
    # Training data (simple examples)
    train_texts = [
        "The cat sat on the mat",
        "The dog ran in the park",
        "A bird flew over the house",
        "The sun shines in the sky",
        "Children play in the garden",
        "Books are on the shelf",
        "Water flows in the river",
        "The car drives on the road"
    ]
    
    # Test data
    test_texts = [
        "The cat played in the garden",      # Similar to training
        "The elephant danced on the moon",   # Unusual combination
        "Quantum mechanics explains physics", # Out-of-domain
        "The the the the the",               # Repetitive
        "Beautiful sunset paints sky orange" # No articles (different style)
    ]
    
    print("=== Custom Perplexity Implementation ===")
    
    # Test different configurations
    configurations = [
        {'n': 2, 'smoothing': 'mle'},
        {'n': 2, 'smoothing': 'add_one'},
        {'n': 3, 'smoothing': 'add_one'}
    ]
    
    all_results = {}
    
    for config in configurations:
        print(f"\\n{'='*60}")
        print(f"Configuration: {config['n']}-gram with {config['smoothing']} smoothing")
        print('='*60)
        
        calculator = CustomPerplexityCalculator(
            smoothing_method=config['smoothing'], 
            vocab_size=100
        )
        
        results, ngram_counts, context_counts = calculator.evaluate_model(
            train_texts, test_texts, n=config['n']
        )
        
        config_key = f"{config['n']}-gram_{config['smoothing']}"
        all_results[config_key] = results
    
    # Compare results
    print(f"\\n{'='*80}")
    print("COMPARISON SUMMARY")
    print('='*80)
    
    comparison_df = []
    for config_key, results in all_results.items():
        for i, result in enumerate(results):
            comparison_df.append({
                'Configuration': config_key,
                'Test_Case': f"Test {i+1}",
                'Text': result['text'][:30] + "...",
                'Perplexity': result['perplexity'],
                'Tokens': result['token_count']
            })
    
    df = pd.DataFrame(comparison_df)
    
    # Pivot table for better visualization
    pivot_df = df.pivot(index='Test_Case', columns='Configuration', values='Perplexity')
    print(pivot_df.round(2))
    
    # Visualization
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # 1. Perplexity comparison across configurations
    test_cases = [f"Test {i+1}" for i in range(len(test_texts))]
    x_pos = np.arange(len(test_cases))
    width = 0.25
    
    configs = list(all_results.keys())
    colors = ['blue', 'green', 'red']
    
    ax1 = axes[0, 0]
    for i, config in enumerate(configs):
        perplexities = [result['perplexity'] for result in all_results[config]]
        ax1.bar(x_pos + i*width, perplexities, width, label=config, alpha=0.8, color=colors[i])
    
    ax1.set_xlabel('Test Cases')
    ax1.set_ylabel('Perplexity (log scale)')
    ax1.set_yscale('log')
    ax1.set_title('Perplexity Comparison Across Configurations')
    ax1.set_xticks(x_pos + width)
    ax1.set_xticklabels(test_cases)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. Effect of smoothing
    ax2 = axes[0, 1]
    mle_perps = [result['perplexity'] for result in all_results['2-gram_mle']]
    smooth_perps = [result['perplexity'] for result in all_results['2-gram_add_one']]
    
    ax2.scatter(mle_perps, smooth_perps, alpha=0.7, s=100)
    ax2.plot([min(mle_perps + smooth_perps), max(mle_perps + smooth_perps)], 
             [min(mle_perps + smooth_perps), max(mle_perps + smooth_perps)], 
             'r--', alpha=0.5)
    ax2.set_xlabel('MLE Perplexity')
    ax2.set_ylabel('Add-One Smoothed Perplexity')
    ax2.set_title('Effect of Smoothing')
    ax2.set_xscale('log')
    ax2.set_yscale('log')
    ax2.grid(True, alpha=0.3)
    
    # 3. N-gram order comparison
    ax3 = axes[1, 0]
    bigram_perps = [result['perplexity'] for result in all_results['2-gram_add_one']]
    trigram_perps = [result['perplexity'] for result in all_results['3-gram_add_one']]
    
    ax3.scatter(bigram_perps, trigram_perps, alpha=0.7, s=100, c=range(len(bigram_perps)), cmap='viridis')
    ax3.plot([min(bigram_perps + trigram_perps), max(bigram_perps + trigram_perps)], 
             [min(bigram_perps + trigram_perps), max(bigram_perps + trigram_perps)], 
             'r--', alpha=0.5)
    ax3.set_xlabel('Bigram Perplexity')
    ax3.set_ylabel('Trigram Perplexity')
    ax3.set_title('Bigram vs Trigram Models')
    ax3.set_xscale('log')
    ax3.set_yscale('log')
    ax3.grid(True, alpha=0.3)
    
    # 4. Token count vs perplexity
    ax4 = axes[1, 1]
    for config in configs:
        tokens = [result['token_count'] for result in all_results[config]]
        perps = [result['perplexity'] for result in all_results[config]]
        ax4.scatter(tokens, perps, label=config, alpha=0.7, s=60)
    
    ax4.set_xlabel('Token Count')
    ax4.set_ylabel('Perplexity')
    ax4.set_title('Token Count vs Perplexity')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return all_results, df

# Run custom perplexity demonstration
custom_results, custom_df = demonstrate_custom_perplexity()

#### 7. Perplexity for Language Model Evaluation {#section7}

Let's evaluate real language models using perplexity with practical examples:

In [ ]:
class LLMPerplexityEvaluator:
    """Evaluate Language Models using perplexity"""
    
    def __init__(self, model_name="gpt2", device="cpu"):
        """Initialize with a specific model"""
        self.model_name = model_name
        self.device = device
        
        try:
            # Load tokenizer and model
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.model = AutoModelForCausalLM.from_pretrained(model_name)
            
            # Set pad token if not available
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token
            
            self.model.eval()  # Set to evaluation mode
            print(f"Loaded model: {model_name}")
            
        except Exception as e:
            print(f"Error loading model {model_name}: {e}")
            print("Using simulated results for demonstration...")
            self.model = None
            self.tokenizer = None
    
    def calculate_perplexity(self, text, max_length=512):
        """Calculate perplexity for a given text"""
        if self.model is None:
            # Return simulated results if model loading failed
            return np.random.uniform(10, 100), len(text.split())
        
        try:
            # Tokenize the text
            encoded = self.tokenizer.encode(text, return_tensors='pt', max_length=max_length, truncation=True)
            
            with torch.no_grad():
                outputs = self.model(encoded, labels=encoded)
                loss = outputs.loss
                perplexity = torch.exp(loss).item()
            
            return perplexity, encoded.shape[1]
            
        except Exception as e:
            print(f"Error calculating perplexity: {e}")
            return float('inf'), 0
    
    def evaluate_texts(self, texts, text_labels=None):
        """Evaluate multiple texts and return results"""
        if text_labels is None:
            text_labels = [f"Text {i+1}" for i in range(len(texts))]
        
        results = []
        
        print(f"Evaluating {len(texts)} texts with {self.model_name}...")
        print("-" * 60)
        
        for text, label in zip(texts, text_labels):
            perplexity, token_count = self.calculate_perplexity(text)
            
            result = {
                'label': label,
                'text': text,
                'perplexity': perplexity,
                'token_count': token_count,
                'words': len(text.split())
            }
            
            results.append(result)
            
            print(f"{label:15}: PPL = {perplexity:7.2f} | Tokens = {token_count:3d} | {text[:50]}...")
        
        return results

def demonstrate_llm_perplexity_evaluation():
    """Demonstrate LLM evaluation using perplexity"""
    
    print("=== LLM Perplexity Evaluation ===")
    
    # Test texts from different domains and quality levels
    test_texts = {
        "High Quality": [
            "The advancement of artificial intelligence has transformed modern technology and continues to shape our future.",
            "Climate change represents one of the most significant challenges facing humanity in the 21st century.",
            "Renewable energy sources like solar and wind power are becoming increasingly cost-effective and efficient."
        ],
        "Medium Quality": [
            "AI technology is changing many things in the world and making life different for people everywhere.",
            "The weather is getting worse because of pollution and other bad things that humans are doing.",
            "Solar panels and wind machines are getting better and cheaper for making clean electricity."
        ],
        "Low Quality": [
            "AI computer smart making world different technology future change happening now yes.",
            "Weather bad pollution humans doing climate change problem very big issue world.",
            "Solar wind electricity clean energy good better cheap making power for people use."
        ],
        "Technical": [
            "Neural networks utilize backpropagation algorithms to optimize gradient descent during training phases.",
            "Photovoltaic cells convert photons into electrical current through the photovoltaic effect mechanism.",
            "Machine learning models demonstrate improved performance through regularization and hyperparameter tuning."
        ],
        "Conversational": [
            "Hey, did you know that AI is getting really good at understanding what we're saying?",
            "I think we really need to do something about climate change before it's too late.",
            "My neighbor just got solar panels and says they're saving a ton on electricity bills."
        ]
    }
    
    # Simulate evaluation with different models (using simplified approach due to resource constraints)
    model_configs = [
        {"name": "GPT-2", "base_ppl": 25},
        {"name": "GPT-2 Medium", "base_ppl": 20},
        {"name": "GPT-2 Large", "base_ppl": 18}
    ]
    
    all_results = {}
    
    # For demonstration, we'll simulate results rather than loading actual models
    print("\\nSimulating perplexity evaluation across different models...")
    print("(In practice, you would load actual models like GPT-2, BERT, etc.)\\n")
    
    for model_config in model_configs:
        model_name = model_config["name"]
        base_ppl = model_config["base_ppl"]
        
        print(f"Model: {model_name}")
        print("-" * 40)
        
        model_results = []
        
        for category, texts in test_texts.items():
            category_perplexities = []
            
            for i, text in enumerate(texts):
                # Simulate perplexity based on text quality and length
                quality_multiplier = {
                    "High Quality": 0.8,
                    "Medium Quality": 1.2,
                    "Low Quality": 2.5,
                    "Technical": 1.5,  # Higher due to specialized vocabulary
                    "Conversational": 0.9
                }[category]
                
                # Add some randomness and length dependency
                length_factor = 1 + (len(text.split()) - 15) * 0.01
                noise = np.random.uniform(0.9, 1.1)
                
                simulated_ppl = base_ppl * quality_multiplier * length_factor * noise
                
                model_results.append({
                    'category': category,
                    'text_id': i + 1,
                    'text': text,
                    'perplexity': simulated_ppl,
                    'token_count': len(text.split()) + 2,  # Approximate
                    'words': len(text.split())
                })
                
                category_perplexities.append(simulated_ppl)
            
            avg_ppl = np.mean(category_perplexities)
            print(f"  {category:15}: Avg PPL = {avg_ppl:6.2f}")
        
        all_results[model_name] = model_results
        print()
    
    # Analysis and visualization
    print("="*60)
    print("DETAILED ANALYSIS")
    print("="*60)
    
    # Convert to DataFrame for analysis
    analysis_data = []
    for model, results in all_results.items():
        for result in results:
            analysis_data.append({
                'Model': model,
                'Category': result['category'],
                'Perplexity': result['perplexity'],
                'Words': result['words'],
                'Text': result['text'][:50] + "..."
            })
    
    df = pd.DataFrame(analysis_data)
    
    # Summary statistics
    summary = df.groupby(['Model', 'Category'])['Perplexity'].agg(['mean', 'std']).round(2)
    print("\\nSummary Statistics (Mean ± Std):")
    print(summary)
    
    # Model comparison
    print("\\n\\nModel Comparison (Overall Average Perplexity):")
    model_comparison = df.groupby('Model')['Perplexity'].mean().sort_values()
    for model, avg_ppl in model_comparison.items():
        print(f"  {model:15}: {avg_ppl:6.2f}")
    
    # Visualizations
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. Perplexity by category and model
    ax1 = axes[0, 0]
    categories = df['Category'].unique()
    x_pos = np.arange(len(categories))
    width = 0.25
    
    models = df['Model'].unique()
    colors = ['blue', 'green', 'red']
    
    for i, model in enumerate(models):
        model_data = df[df['Model'] == model]
        avg_perplexities = [model_data[model_data['Category'] == cat]['Perplexity'].mean() 
                           for cat in categories]
        ax1.bar(x_pos + i*width, avg_perplexities, width, label=model, alpha=0.8, color=colors[i])
    
    ax1.set_xlabel('Text Categories')
    ax1.set_ylabel('Average Perplexity')
    ax1.set_title('Perplexity by Category and Model')
    ax1.set_xticks(x_pos + width)
    ax1.set_xticklabels(categories, rotation=45)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. Perplexity distribution
    ax2 = axes[0, 1]
    for model in models:
        model_perplexities = df[df['Model'] == model]['Perplexity']
        ax2.hist(model_perplexities, bins=10, alpha=0.6, label=model, density=True)
    
    ax2.set_xlabel('Perplexity')
    ax2.set_ylabel('Density')
    ax2.set_title('Perplexity Distribution by Model')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. Text length vs perplexity
    ax3 = axes[1, 0]
    for model in models:
        model_data = df[df['Model'] == model]
        ax3.scatter(model_data['Words'], model_data['Perplexity'], 
                   label=model, alpha=0.6, s=50)
    
    ax3.set_xlabel('Text Length (words)')
    ax3.set_ylabel('Perplexity')
    ax3.set_title('Text Length vs Perplexity')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # 4. Category performance heatmap
    ax4 = axes[1, 1]
    pivot_data = df.pivot_table(values='Perplexity', index='Category', columns='Model', aggfunc='mean')
    sns.heatmap(pivot_data, annot=True, cmap='YlOrRd_r', ax=ax4, fmt='.1f')
    ax4.set_title('Perplexity Heatmap (Lower is Better)')
    
    plt.tight_layout()
    plt.show()
    
    return df, all_results

# Run LLM perplexity evaluation
llm_eval_df, llm_results = demonstrate_llm_perplexity_evaluation()

#### 8. Perplexity in Text Generation Quality {#section8}

Let's explore how perplexity relates to text generation quality and human perception:

In [ ]:
def analyze_generation_quality_with_perplexity():
    """Analyze the relationship between perplexity and generation quality"""
    
    print("=== Perplexity vs Text Generation Quality Analysis ===")
    
    # Generated text samples with different characteristics
    generation_samples = [
        {
            "method": "Greedy Decoding",
            "text": "The cat sat on the mat. The cat sat on the mat. The cat sat on the mat.",
            "perplexity": 8.5,  # Low due to repetition
            "human_rating": 2.1,
            "fluency": 4.0,
            "diversity": 1.0,
            "coherence": 2.5
        },
        {
            "method": "Top-k Sampling (k=5)",
            "text": "The curious cat explored the garden, discovering hidden flowers among the green leaves.",
            "perplexity": 15.2,
            "human_rating": 4.2,
            "fluency": 4.5,
            "diversity": 4.0,
            "coherence": 4.3
        },
        {
            "method": "Top-p Sampling (p=0.9)",
            "text": "A magnificent feline wandered through the botanical paradise, uncovering secret blossoms.",
            "perplexity": 22.8,
            "human_rating": 3.8,
            "fluency": 4.2,
            "diversity": 4.5,
            "coherence": 3.9
        },
        {
            "method": "High Temperature (T=1.5)",
            "text": "Cat mysterious garden blooming adventures whiskers sunlight dancing petals everywhere beautiful.",
            "perplexity": 45.3,
            "human_rating": 2.8,
            "fluency": 2.5,
            "diversity": 4.8,
            "coherence": 2.2
        },
        {
            "method": "Very High Temperature (T=2.0)",
            "text": "Elephant keyboard symphony mathematics purple jumping theoretical quantum breakfast revolution.",
            "perplexity": 125.7,
            "human_rating": 1.2,
            "fluency": 1.5,
            "diversity": 5.0,
            "coherence": 1.0
        },
        {
            "method": "Beam Search (beam=5)",
            "text": "The cat sat in the garden and looked at the beautiful flowers in the sunlight.",
            "perplexity": 12.1,
            "human_rating": 3.9,
            "fluency": 4.8,
            "diversity": 2.8,
            "coherence": 4.5
        }
    ]
    
    # Convert to DataFrame
    df = pd.DataFrame(generation_samples)
    
    print("Generation Quality Analysis:")
    print("-" * 50)
    for _, sample in df.iterrows():
        print(f"Method: {sample['method']}")
        print(f"  Text: {sample['text']}")
        print(f"  Perplexity: {sample['perplexity']:6.1f}")
        print(f"  Human Rating: {sample['human_rating']:4.1f}/5.0")
        print(f"  Fluency: {sample['fluency']:4.1f}")
        print(f"  Diversity: {sample['diversity']:4.1f}")
        print(f"  Coherence: {sample['coherence']:4.1f}")
        print()
    
    # Calculate correlations
    print("Correlation Analysis:")
    print("-" * 30)
    correlations = {
        'Perplexity vs Human Rating': df['perplexity'].corr(df['human_rating']),
        'Perplexity vs Fluency': df['perplexity'].corr(df['fluency']),
        'Perplexity vs Diversity': df['perplexity'].corr(df['diversity']),
        'Perplexity vs Coherence': df['perplexity'].corr(df['coherence'])
    }
    
    for metric, corr in correlations.items():
        direction = "↑" if corr > 0 else "↓"
        strength = "Strong" if abs(corr) > 0.7 else "Moderate" if abs(corr) > 0.4 else "Weak"
        print(f"{metric:25}: {corr:6.3f} {direction} ({strength})")
    
    # Temperature vs Perplexity analysis
    print("\\n" + "="*60)
    print("TEMPERATURE EFFECT ANALYSIS")
    print("="*60)
    
    temperature_data = [
        {"temp": 0.1, "perplexity": 8.2, "quality": "Repetitive but fluent"},
        {"temp": 0.5, "perplexity": 12.5, "quality": "Good balance"},
        {"temp": 0.8, "perplexity": 18.3, "quality": "Natural and diverse"},
        {"temp": 1.0, "perplexity": 25.1, "quality": "More creative"},
        {"temp": 1.2, "perplexity": 35.8, "quality": "Sometimes incoherent"},
        {"temp": 1.5, "perplexity": 52.4, "quality": "Often nonsensical"},
        {"temp": 2.0, "perplexity": 98.7, "quality": "Random word salad"}
    ]
    
    temp_df = pd.DataFrame(temperature_data)
    
    for _, row in temp_df.iterrows():
        print(f"Temperature {row['temp']:3.1f}: PPL = {row['perplexity']:5.1f} | {row['quality']}")
    
    # Visualization
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # 1. Perplexity vs Human Rating
    ax1 = axes[0, 0]
    ax1.scatter(df['perplexity'], df['human_rating'], s=100, alpha=0.7, c='blue')
    
    # Add method labels
    for _, row in df.iterrows():
        ax1.annotate(row['method'].split()[0], 
                    (row['perplexity'], row['human_rating']),
                    xytext=(5, 5), textcoords='offset points', fontsize=9)
    
    ax1.set_xlabel('Perplexity')
    ax1.set_ylabel('Human Rating')
    ax1.set_title('Perplexity vs Human Rating')
    ax1.grid(True, alpha=0.3)
    
    # Trend line
    z = np.polyfit(df['perplexity'], df['human_rating'], 1)
    p = np.poly1d(z)
    ax1.plot(df['perplexity'], p(df['perplexity']), "r--", alpha=0.8)
    
    # 2. Multi-dimensional quality analysis
    ax2 = axes[0, 1]
    qualities = ['fluency', 'diversity', 'coherence']
    x_pos = np.arange(len(df))
    width = 0.25
    
    for i, quality in enumerate(qualities):
        ax2.bar(x_pos + i*width, df[quality], width, alpha=0.8, label=quality.capitalize())
    
    ax2.set_xlabel('Generation Methods')
    ax2.set_ylabel('Quality Score')
    ax2.set_title('Multi-dimensional Quality Analysis')
    ax2.set_xticks(x_pos + width)
    ax2.set_xticklabels([m.split()[0] for m in df['method']], rotation=45)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. Temperature effect
    ax3 = axes[0, 2]
    ax3.plot(temp_df['temp'], temp_df['perplexity'], 'bo-', linewidth=2, markersize=8)
    ax3.set_xlabel('Temperature')
    ax3.set_ylabel('Perplexity')
    ax3.set_title('Temperature vs Perplexity')
    ax3.grid(True, alpha=0.3)
    ax3.set_yscale('log')
    
    # 4. Quality trade-offs
    ax4 = axes[1, 0]
    ax4.scatter(df['fluency'], df['diversity'], s=df['perplexity']*3, 
               alpha=0.6, c=df['human_rating'], cmap='RdYlGn')
    ax4.set_xlabel('Fluency')
    ax4.set_ylabel('Diversity')
    ax4.set_title('Fluency vs Diversity Trade-off\\n(Size=Perplexity, Color=Human Rating)')
    
    # Add colorbar
    cbar = plt.colorbar(ax4.collections[0], ax=ax4)
    cbar.set_label('Human Rating')
    
    # 5. Perplexity distribution by method
    ax5 = axes[1, 1]
    methods = df['method'].str.split().str[0]  # Get first word of method
    perplexities = df['perplexity']
    
    colors = plt.cm.viridis(np.linspace(0, 1, len(methods)))
    bars = ax5.bar(methods, perplexities, color=colors, alpha=0.8)
    
    ax5.set_ylabel('Perplexity')
    ax5.set_title('Perplexity by Generation Method')
    ax5.tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for bar, val in zip(bars, perplexities):
        ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, 
                f'{val:.1f}', ha='center', va='bottom')
    
    # 6. Correlation heatmap
    ax6 = axes[1, 2]
    corr_data = df[['perplexity', 'human_rating', 'fluency', 'diversity', 'coherence']].corr()
    sns.heatmap(corr_data, annot=True, cmap='coolwarm', center=0, ax=ax6, fmt='.2f')
    ax6.set_title('Quality Metrics Correlation')
    
    plt.tight_layout()
    plt.show()
    
    return df, temp_df

# Run generation quality analysis
quality_df, temp_analysis_df = analyze_generation_quality_with_perplexity()

#### 10. Comparing Multiple LLMs with Perplexity {#section10}

Let's create a comprehensive evaluation pipeline to compare multiple LLMs using perplexity:

In [ ]:
class LLMPerplexityBenchmark:
    """Comprehensive LLM comparison using perplexity"""
    
    def __init__(self):
        self.results = []
        self.test_datasets = self._create_test_datasets()
    
    def _create_test_datasets(self):
        """Create diverse test datasets for evaluation"""
        return {
            "News": [
                "The latest breakthrough in artificial intelligence has revolutionized medical diagnosis.",
                "Economic indicators suggest a potential recession in the coming quarter.",
                "Climate scientists warn of accelerating ice cap melting in Antarctica."
            ],
            "Literature": [
                "The old man sat by the window, watching raindrops race down the glass.",
                "In the depths of winter, she found within herself an invincible summer.",
                "Time passed slowly in the quiet library, where dust motes danced in sunbeams."
            ],
            "Technical": [
                "The neural network implements backpropagation using gradient descent optimization.",
                "Quantum entanglement enables instantaneous information transfer between particles.",
                "Docker containers provide isolated environments for microservice deployment."
            ],
            "Conversational": [
                "Hey, how was your day? Mine was pretty crazy with all the meetings.",
                "I can't believe it's already Friday! This week went by so fast.",
                "Did you see that new movie? I heard it's really good but also quite long."
            ],
            "Creative": [
                "The purple elephant danced with silver stars under a chocolate moon.",
                "Whispered secrets of ancient trees echoed through the emerald forest.",
                "Dreams painted themselves across the canvas of sleeping minds."
            ]
        }
    
    def simulate_model_evaluation(self, model_configs):
        """Simulate evaluation of multiple models"""
        
        print("=== LLM Perplexity Benchmark ===")
        print(f"Evaluating {len(model_configs)} models on {len(self.test_datasets)} domains")
        print("=" * 60)
        
        # Simulate model performance characteristics
        np.random.seed(42)
        
        all_results = []
        
        for model_config in model_configs:
            model_name = model_config['name']
            base_perplexity = model_config['base_ppl']
            domain_strengths = model_config.get('domain_strengths', {})
            
            print(f"\\nEvaluating {model_name}...")
            print("-" * 30)
            
            model_results = {'model': model_name, 'domains': {}}
            
            for domain, texts in self.test_datasets.items():
                domain_modifier = domain_strengths.get(domain, 1.0)
                domain_perplexities = []
                
                for text in texts:
                    # Simulate perplexity based on text characteristics
                    text_length = len(text.split())
                    complexity_factor = 1 + (text_length - 10) * 0.01
                    
                    # Add some randomness
                    noise = np.random.uniform(0.85, 1.15)
                    
                    perplexity = base_perplexity * domain_modifier * complexity_factor * noise
                    domain_perplexities.append(perplexity)
                    
                    all_results.append({
                        'model': model_name,
                        'domain': domain,
                        'text': text,
                        'perplexity': perplexity,
                        'text_length': text_length
                    })
                
                avg_domain_ppl = np.mean(domain_perplexities)
                model_results['domains'][domain] = avg_domain_ppl
                
                print(f"  {domain:15}: {avg_domain_ppl:6.2f} ± {np.std(domain_perplexities):5.2f}")
            
            # Calculate overall performance
            overall_ppl = np.mean(list(model_results['domains'].values()))
            model_results['overall'] = overall_ppl
            print(f"  {'Overall':15}: {overall_ppl:6.2f}")
        
        return pd.DataFrame(all_results)
    
    def statistical_analysis(self, results_df):
        """Perform statistical analysis of results"""
        
        print("\\n" + "=" * 60)
        print("STATISTICAL ANALYSIS")
        print("=" * 60)
        
        # 1. Overall model ranking
        print("\\n1. OVERALL MODEL RANKING")
        print("-" * 30)
        
        overall_ranking = results_df.groupby('model')['perplexity'].mean().sort_values()
        for rank, (model, avg_ppl) in enumerate(overall_ranking.items(), 1):
            print(f"{rank}. {model:15}: {avg_ppl:6.2f}")
        
        # 2. Domain-specific analysis
        print("\\n2. DOMAIN-SPECIFIC PERFORMANCE")
        print("-" * 35)
        
        domain_analysis = results_df.pivot_table(
            values='perplexity', 
            index='domain', 
            columns='model', 
            aggfunc='mean'
        ).round(2)
        
        print(domain_analysis)
        
        # 3. Statistical significance testing
        print("\\n3. STATISTICAL SIGNIFICANCE TESTS")
        print("-" * 40)
        
        models = results_df['model'].unique()
        
        if len(models) >= 2:
            from itertools import combinations
            
            print("Pairwise t-tests (p-values):")
            for model1, model2 in combinations(models, 2):
                scores1 = results_df[results_df['model'] == model1]['perplexity']
                scores2 = results_df[results_df['model'] == model2]['perplexity']
                
                _, p_value = stats.ttest_ind(scores1, scores2)
                significance = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else "ns"
                
                print(f"  {model1:12} vs {model2:12}: p = {p_value:.4f} {significance}")
        
        # 4. Variance analysis
        print("\\n4. VARIANCE ANALYSIS")
        print("-" * 25)
        
        variance_analysis = results_df.groupby('model')['perplexity'].agg(['mean', 'std', 'min', 'max']).round(2)
        print(variance_analysis)
        
        return domain_analysis, overall_ranking
    
    def create_comprehensive_visualizations(self, results_df):
        """Create comprehensive visualizations"""
        
        fig = plt.figure(figsize=(20, 16))
        gs = fig.add_gridspec(4, 3, hspace=0.3, wspace=0.3)
        
        models = results_df['model'].unique()
        domains = results_df['domain'].unique()
        colors = plt.cm.Set3(np.linspace(0, 1, len(models)))
        
        # 1. Overall performance comparison
        ax1 = fig.add_subplot(gs[0, 0])
        overall_perf = results_df.groupby('model')['perplexity'].mean().sort_values()
        bars = ax1.bar(range(len(overall_perf)), overall_perf.values, 
                      color=colors[:len(overall_perf)], alpha=0.8)
        
        ax1.set_xlabel('Models')
        ax1.set_ylabel('Average Perplexity')
        ax1.set_title('Overall Model Performance')
        ax1.set_xticks(range(len(overall_perf)))
        ax1.set_xticklabels(overall_perf.index, rotation=45)
        
        # Add value labels
        for bar, val in zip(bars, overall_perf.values):
            ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
                    f'{val:.1f}', ha='center', va='bottom')
        
        # 2. Domain-specific performance heatmap
        ax2 = fig.add_subplot(gs[0, 1])
        pivot_data = results_df.pivot_table(values='perplexity', index='domain', columns='model', aggfunc='mean')
        sns.heatmap(pivot_data, annot=True, cmap='YlOrRd', ax=ax2, fmt='.1f')
        ax2.set_title('Domain-specific Performance Heatmap')
        
        # 3. Performance distribution
        ax3 = fig.add_subplot(gs[0, 2])
        for i, model in enumerate(models):
            model_data = results_df[results_df['model'] == model]['perplexity']
            ax3.hist(model_data, bins=10, alpha=0.6, label=model, color=colors[i], density=True)
        
        ax3.set_xlabel('Perplexity')
        ax3.set_ylabel('Density')
        ax3.set_title('Perplexity Distribution by Model')
        ax3.legend()
        
        # 4. Box plot comparison
        ax4 = fig.add_subplot(gs[1, 0])
        results_df.boxplot(column='perplexity', by='model', ax=ax4)
        ax4.set_title('Perplexity Distribution Comparison')
        ax4.set_xlabel('Model')
        plt.suptitle('')  # Remove automatic title
        
        # 5. Domain performance radar chart
        ax5 = fig.add_subplot(gs[1, 1], projection='polar')
        
        angles = np.linspace(0, 2 * np.pi, len(domains), endpoint=False).tolist()
        angles += angles[:1]  # Complete the circle
        
        for i, model in enumerate(models):
            model_domain_scores = []
            for domain in domains:
                score = results_df[(results_df['model'] == model) & 
                                 (results_df['domain'] == domain)]['perplexity'].mean()
                # Invert scores for radar chart (higher is better visually)
                inverted_score = 100 / score  # Simple inversion
                model_domain_scores.append(inverted_score)
            
            model_domain_scores += model_domain_scores[:1]  # Complete the circle
            
            ax5.plot(angles, model_domain_scores, 'o-', linewidth=2, 
                    label=model, color=colors[i])
            ax5.fill(angles, model_domain_scores, alpha=0.1, color=colors[i])
        
        ax5.set_xticks(angles[:-1])
        ax5.set_xticklabels(domains)
        ax5.set_title('Domain Performance Profile\\n(Higher = Better)')
        ax5.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
        
        # 6. Text length vs perplexity
        ax6 = fig.add_subplot(gs[1, 2])
        for i, model in enumerate(models):
            model_data = results_df[results_df['model'] == model]
            ax6.scatter(model_data['text_length'], model_data['perplexity'], 
                       alpha=0.6, label=model, color=colors[i], s=50)
        
        ax6.set_xlabel('Text Length (words)')
        ax6.set_ylabel('Perplexity')
        ax6.set_title('Text Length vs Perplexity')
        ax6.legend()
        
        # 7. Performance consistency
        ax7 = fig.add_subplot(gs[2, 0])
        consistency_data = results_df.groupby('model')['perplexity'].std().sort_values()
        bars = ax7.bar(range(len(consistency_data)), consistency_data.values, 
                      color='orange', alpha=0.8)
        
        ax7.set_xlabel('Models')
        ax7.set_ylabel('Standard Deviation')
        ax7.set_title('Performance Consistency\\n(Lower = More Consistent)')
        ax7.set_xticks(range(len(consistency_data)))
        ax7.set_xticklabels(consistency_data.index, rotation=45)
        
        # 8. Domain difficulty ranking
        ax8 = fig.add_subplot(gs[2, 1])
        domain_difficulty = results_df.groupby('domain')['perplexity'].mean().sort_values(ascending=False)
        bars = ax8.bar(range(len(domain_difficulty)), domain_difficulty.values, 
                      color='lightcoral', alpha=0.8)
        
        ax8.set_xlabel('Domains')
        ax8.set_ylabel('Average Perplexity')
        ax8.set_title('Domain Difficulty Ranking')
        ax8.set_xticks(range(len(domain_difficulty)))
        ax8.set_xticklabels(domain_difficulty.index, rotation=45)
        
        # 9. Model performance matrix
        ax9 = fig.add_subplot(gs[2, 2])
        
        # Create performance matrix (rank-based)
        performance_matrix = np.zeros((len(models), len(domains)))
        
        for j, domain in enumerate(domains):
            domain_data = results_df[results_df['domain'] == domain]
            domain_ranking = domain_data.groupby('model')['perplexity'].mean().rank()
            
            for i, model in enumerate(models):
                if model in domain_ranking.index:
                    performance_matrix[i, j] = domain_ranking[model]
        
        im = ax9.imshow(performance_matrix, cmap='RdYlGn_r', aspect='auto')
        ax9.set_xticks(range(len(domains)))
        ax9.set_xticklabels(domains, rotation=45)
        ax9.set_yticks(range(len(models)))
        ax9.set_yticklabels(models)
        ax9.set_title('Performance Ranking Matrix\\n(1=Best, Higher=Worse)')
        
        # Add colorbar
        plt.colorbar(im, ax=ax9, shrink=0.8)
        
        # 10. Summary statistics table
        ax10 = fig.add_subplot(gs[3, :])
        ax10.axis('tight')
        ax10.axis('off')
        
        summary_stats = results_df.groupby('model')['perplexity'].agg([
            'count', 'mean', 'std', 'min', 'max'
        ]).round(2)
        
        table = ax10.table(cellText=summary_stats.values,
                          rowLabels=summary_stats.index,
                          colLabels=summary_stats.columns,
                          cellLoc='center',
                          loc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1.2, 1.5)
        ax10.set_title('Summary Statistics', pad=20)
        
        plt.show()

def run_comprehensive_llm_comparison():
    """Run comprehensive LLM comparison using perplexity"""
    
    # Define model configurations
    model_configs = [
        {
            'name': 'GPT-4',
            'base_ppl': 18.5,
            'domain_strengths': {
                'News': 0.8, 'Literature': 0.9, 'Technical': 0.9, 
                'Conversational': 0.7, 'Creative': 0.8
            }
        },
        {
            'name': 'GPT-3.5',
            'base_ppl': 22.3,
            'domain_strengths': {
                'News': 0.9, 'Literature': 1.0, 'Technical': 1.1, 
                'Conversational': 0.8, 'Creative': 0.9
            }
        },
        {
            'name': 'Claude-2',
            'base_ppl': 20.1,
            'domain_strengths': {
                'News': 0.85, 'Literature': 0.8, 'Technical': 0.95, 
                'Conversational': 0.75, 'Creative': 0.85
            }
        },
        {
            'name': 'PaLM-2',
            'base_ppl': 24.7,
            'domain_strengths': {
                'News': 0.95, 'Literature': 1.1, 'Technical': 0.8, 
                'Conversational': 0.9, 'Creative': 1.0
            }
        },
        {
            'name': 'LLaMA-2-70B',
            'base_ppl': 26.9,
            'domain_strengths': {
                'News': 1.0, 'Literature': 1.05, 'Technical': 1.15, 
                'Conversational': 0.95, 'Creative': 1.1
            }
        }
    ]
    
    # Run benchmark
    benchmark = LLMPerplexityBenchmark()
    results_df = benchmark.simulate_model_evaluation(model_configs)
    
    # Perform analysis
    domain_analysis, overall_ranking = benchmark.statistical_analysis(results_df)
    
    # Create visualizations
    benchmark.create_comprehensive_visualizations(results_df)
    
    return results_df, domain_analysis, overall_ranking

# Run comprehensive LLM comparison
final_results, domain_perf, model_ranking = run_comprehensive_llm_comparison()

**Conclusion and Best Practices**

Perplexity is a powerful metric for evaluating language models, offering both intrinsic quality assessment and comparative analysis capabilities. Here are the key takeaways:

**When to Use Perplexity:**
- Comparing different language models
- Evaluating model performance across domains
- Assessing text generation quality
- Pre-training and fine-tuning evaluation
- Domain adaptation effectiveness

**When to Supplement with Other Metrics:**
- Task-specific evaluation (combine with ROUGE, BLEU)
- Human preference assessment
- Factual accuracy evaluation
- Bias and fairness analysis

**Best Practices:**
- Use consistent evaluation datasets
- Consider domain-specific variations
- Combine with human evaluation
- Report confidence intervals
- Analyze statistical significance
- Consider computational efficiency

**Implementation Tips:**
- Handle out-of-vocabulary tokens appropriately
- Use sliding window for long texts
- Consider weighted perplexity for importance
- Implement efficient batching
- Cache model computations

This comprehensive guide provides the foundation for implementing perplexity-based evaluation in your LLM applications. Remember that perplexity is one component of a robust evaluation framework.

**References:**
- Jelinek, F. (1997). Statistical Methods for Speech Recognition
- Chen, S. F., & Goodman, J. (1999). An empirical study of smoothing techniques
- Bengio, Y. (2003). Neural probabilistic language models
- Radford, A. (2019). Language Models are Unsupervised Multitask Learners